# Install

```shell
pip install -U langchain
# Requires Python 3.10+
```


# Quick Start


    This quickstart takes you from a simple setup to a fully functional AI agent in just a few minutes.

### Build a basic agent

Start by creating a simple agent that can answer questions and call tools. The agent will use Qwen3 as its language model, a basic weather function as a tool, and a simple prompt to guide its behavior.

In [1]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

qwen_model = ChatOpenAI(
    model="qwen3-max",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key="sk-8b293e101d5e41f1b3116d9848b7ac59"
)

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=qwen_model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='d40509e1-f8b6-41e7-bf8f-9eef00383b11'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 265, 'total_tokens': 286, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen3-max', 'system_fingerprint': None, 'id': 'chatcmpl-a1b2cd73-e17e-9d85-a675-793ae135c681', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d1f0a-e99f-7bf1-bb08-e328d6ae1a7d-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'sf'}, 'id': 'call_1681f9d742134c078fbd6c0c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 265, 'output_tokens': 21, 'total_tokens': 286, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
  ToolMessage(content="It's always sunny i

### Build a real-world agent

    Next, build a practical weather forecasting agent that demonstrates key production concepts:
1. Detailed system prompts for better agent behavior
2. Create tools that integrate with external data
3. Model configuration for consistent responses
4. Structured output for predictable results
5. Conversational memory for chat-like interactions
6. Create and run the agent create a fully functional agent

Let’s walk through each step:

In [2]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver


# Define system prompt
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

# Define context schema
@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

# Define tools
@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

# Configure model
qwen_model = ChatOpenAI(
    model="qwen3-max",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key="sk-8b293e101d5e41f1b3116d9848b7ac59"
)

# Define response format
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None

# Set up memory
checkpointer = InMemorySaver()

# Create agent
agent = create_agent(
    model=qwen_model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ResponseFormat,
    checkpointer=checkpointer
)

# Run agent
# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )


# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     weather_conditions=None
# )

Deserializing unregistered type __main__.ResponseFormat from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('__main__', 'ResponseFormat')]


ResponseFormat(punny_response='Looks like Florida is keeping things sunny-side up—no clouds allowed in this sunshine state!', weather_conditions='sunny')
ResponseFormat(punny_response="You're very welcome! Stay bright and don’t let a single cloud rain on your parade!", weather_conditions='sunny')
